# Рад са моделима из Meta породице

## Увод

Овај час ће обухватити:

- Истраживање два главна модела из Meta породице - Llama 3.1 и Llama 3.2, као и тренутно доступних алтернатива коришћених у примерима кода (Llama 3.3 и Llama 4)
- Разумевање употребних случајева и сценарија за сваки модел
- Пример кода који приказује јединствене карактеристике сваког модела


## Meta породица модела

У овом часу ћемо истражити 2 модела из Meta породице или "Llama стада" - Llama 3.1 и Llama 3.2. Покретни примери кода користе Llama 3.3 и Llama 4, који су тренутно доступне алтернативе у Microsoft Foundry Models каталогу.

Ови модели долазе у различитим варијантама и доступни су у [Microsoft Foundry Models каталогу](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Напомена:** GitHub Models се повлачи крајем јула 2026. Ево више детаља о коришћењу [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) за прототиповање са AI моделима.

> **Напомена:** Конкретне верзије модела на које је овај час првобитно био усмерен (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) су од тада повучене из Microsoft Foundry Models каталога. Примери кода испод користе `Llama-3.3-70B-Instruct` и `Llama-4-Scout-17B-16E-Instruct` као тренутно доступне алтернативе у Microsoft Foundry Models каталогу, тако да се и даље могу користити.

Варијанте модела (као што је првобитно обухваћено у овом часу):
- Llama 3.1 - 70B Instruct (повучено)
- Llama 3.1 - 405B Instruct (повучено)
- Llama 3.2 - 11B Vision Instruct (повучено)
- Llama 3.2 - 90B Vision Instruct (повучено)

Тренутно доступне алтернативе коришћене у примерима кода:
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`)
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`)

*Напомена: Варијанте Llama 3 горе наведене су задржане за историјски контекст и више нису доступне у Microsoft Foundry Models.*



## Ллама 3.1 

Са 405 милијарди параметара, Ллама 3.1 спада у категорију отворених LLM модела. 

Овај модел је унапређење у односу на претходно издање Ллама 3 пружајући: 

- Веће прозоре контекста - 128к токена уместо 8к токена 
- Већи максимални број излазних токена - 4096 уместо 2048 
- Бољу мултилингвалну подршку - због повећања броја токена за тренирање 

Ово омогућава Ллама 3.1 да обрађује сложеније случајеве употребе при изградњи ГенАИ апликација, укључујући: 
- Нативно позивање функција - могућност позивања екстерних алата и функција ван LLM тока рада
- Боље RAG перформансе - услед већег прозора контекста 
- Генерација синтетичких података - могућност стварања ефикасних података за задатке као што је финетјунинг 



### Позивање нативних функција 

Llama 3.1 је фино подешен да буде ефикаснији у позиву функција или алата. Такође има два уграђена алата која модел може препознати као потребна за коришћење на основу упита корисника. Та два алата су: 

- **Brave Search** - Може се користити за добијање најновијих информација као што је време тако што извршава претрагу веба 
- **Wolfram Alpha** - Може се користити за сложеније математичке прорачуне, тако да није потребно писати властите функције. 

Такође можете креирати своје прилагођене алате које LLM може позвати. 

У примеру кода испод: 

- Дефинишемо доступне алате (brave_search, wolfram_alpha) у системском упиту. 
- Шаљемо кориснички упит који пита о времену у одређеном граду. 
- LLM ће одговорити позивом алата Brave Search који ће изгледати овако `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Напомена: овај пример само врши позив алата, ако желите да добијете резултате, мораћете да креирате бесплатан налог на Brave API страници и дефинишете саму функцију` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Иако је ЛЛМ, једно ограничење које Llama 3.1 има је мултимодалност. То јест, могућност коришћења различитих типова уноса као што су слике као упити и пружање одговора. Ова способност је једна од главних карактеристика Llama 3.2. Ове карактеристике такође укључују: 

- Мултимодалност - има способност да оцени и текстуалне и сликовне упите 
- Варијације мале до средње величине (11B и 90B) - ово пружа флексибилне опције распореда, 
- Само текстуалне варијације (1B и 3B) - ово омогућава да се модел распореди на edge / мобилним уређајима и пружа малу латенцију 

Подршка мултимодалности представља велики корак у свету модела отвореног кода. Код пример испод узима и слику и текстуални упит да добије анализу слике. Пошто је Llama 3.2 90B Vision Instruct повучен, пример користи `Llama-4-Scout-17B-16E-Instruct`, тренутно доступну нативно мултимодалну алтернативу. 

### Подршка мултимодалности са Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## Учење не престаје овде, наставите путовање

Након завршетка ове лекције, погледајте нашу [Збирку за учење генеративне вештачке интелигенције](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) да бисте наставили да унапређујете своје знање о генеративној вештачкој интелигенцији!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
